In [ ]:
# ============================================================
# PNEUMONIA DETECTION — SVM  (RBF Kernel)
# Dataset : COVID-19 Radiography Database (Kaggle)
# Classes : Viral Pneumonia (1) vs Normal (0)
#
# Hyperparameters : kernel=rbf | C=1 | gamma=scale
# Scaler          : StandardScaler (mandatory)
# Split           : Train 70% | Validation 15% | Test 15%
#                   (fixed seed — same split for all models)
#
# Training loop   : Up to 20 rounds of stratified re-shuffling
#                   Early stop when Val-F1 does NOT improve
#                   for 5 consecutive rounds
#
# Anti-overfit    : Train/Val gap monitored every round;
#                   if gap > OVERFIT_THRESHOLD the round is
#                   flagged and the model is NOT accepted
#
# Test set        : LOCKED — evaluated only once at the end
#
# Metrics         : Accuracy, Precision, Recall (Sensitivity),
#                   Specificity, F1-Score, ROC-AUC
# Graphs          : Confusion Matrix Heatmap, ROC Curve,
#                   Learning Curve (train vs val F1 per round),
#                   5-Fold CV bar chart, Metrics Summary
# ============================================================

import os, cv2, pickle, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from datetime import datetime
from copy import deepcopy

from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, cross_val_score)
from sklearn.metrics         import (accuracy_score, precision_score,
                                     recall_score, f1_score, roc_auc_score,
                                     roc_curve, confusion_matrix,
                                     classification_report)
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
_BASE = ('/kaggle/input/datasets/tawsifurrahman/'
         'covid19-radiography-database/'
         'COVID-19_Radiography_Dataset')

PNEUMONIA_PATH = os.path.join(_BASE, 'Viral Pneumonia', 'images')
NORMAL_PATH    = os.path.join(_BASE, 'Normal', 'images')

# ============================================================
# CONFIGURATION
# ============================================================
IMG_SIZE          = (128, 128)
RANDOM_STATE      = 42          # fixed seed — same split every model
TRAIN_RATIO       = 0.70
VAL_RATIO         = 0.15
TEST_RATIO        = 0.15
N_COMPONENTS      = 100         # PCA components

# SVM hyperparameters
SVM_KERNEL        = 'rbf'
SVM_C             = 1.0
SVM_GAMMA         = 'scale'

# Training-loop settings
MAX_LOOPS         = 20          # maximum training rounds
PATIENCE          = 5           # early-stop after N rounds with no improvement
OVERFIT_THRESHOLD = 0.10        # flag round if train_f1 - val_f1 > this

N_FOLDS           = 5
OUT               = '/kaggle/working'

print("="*65)
print("  PNEUMONIA DETECTION — SVM (RBF)")
print("  COVID-19 Radiography Database")
print("="*65)
print(f"\n  Kernel : {SVM_KERNEL}  |  C={SVM_C}  |  gamma={SVM_GAMMA}")
print(f"  Max loops : {MAX_LOOPS}  |  Patience : {PATIENCE}")
print(f"  Overfit threshold : train-val F1 gap > {OVERFIT_THRESHOLD}")
print(f"\n  Pneumonia path : {PNEUMONIA_PATH}")
print(f"  Normal path    : {NORMAL_PATH}")

# ============================================================
# HELPERS — DATA
# ============================================================

def load_images(folder_path, label):
    images, labels = [], []
    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"  Found {len(files):>6} images → {folder_path}")
    for fname in tqdm(files, desc="  Loading", leave=False):
        try:
            img = cv2.imread(os.path.join(folder_path, fname),
                             cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, IMG_SIZE)
                images.append(img)
                labels.append(label)
        except Exception:
            pass
    return images, labels


def extract_features(images):
    out = []
    for img in tqdm(images, desc="  Features", leave=False):
        flat  = img.flatten().astype(np.float32)
        stats = np.array([img.mean(), img.std(),
                          img.min(),  img.max()], dtype=np.float32)
        out.append(np.concatenate([flat, stats]))
    return np.array(out)


def make_split(X, y, val_size, test_size, seed):
    """Reproducible stratified 70/15/15 split for a given seed."""
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=(val_size + test_size),
        stratify=y, random_state=seed)
    half = test_size / (val_size + test_size)
    X_v, X_te, y_v, y_te = train_test_split(
        X_tmp, y_tmp, test_size=half,
        stratify=y_tmp, random_state=seed)
    return X_tr, X_v, X_te, y_tr, y_v, y_te


def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return dict(
        accuracy           = float(accuracy_score(y_true, y_pred)),
        precision          = float(precision_score(y_true, y_pred, zero_division=0)),
        recall_sensitivity = float(recall_score(y_true, y_pred, zero_division=0)),
        specificity        = float(tn / (tn + fp)) if (tn + fp) else 0.0,
        f1_score           = float(f1_score(y_true, y_pred, zero_division=0)),
        roc_auc            = float(roc_auc_score(y_true, y_prob)),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))


def print_metrics_table(m, title=""):
    pad = max(0, 48 - len(title))
    print(f"\n  ── {title} {'─'*pad}")
    print(f"  {'Metric':<28} {'Score':>8}   {'%':>7}")
    print(f"  {'─'*48}")
    rows = [('Accuracy',             'accuracy'),
            ('Precision',            'precision'),
            ('Recall (Sensitivity)', 'recall_sensitivity'),
            ('Specificity',          'specificity'),
            ('F1-Score',             'f1_score'),
            ('ROC-AUC',              'roc_auc')]
    for name, key in rows:
        v = m[key]
        print(f"  {name:<28} {v:>8.4f}   {v*100:>6.2f}%")
    print(f"  {'─'*48}")
    print(f"  TP={m['tp']}  TN={m['tn']}  FP={m['fp']}  FN={m['fn']}")


# ============================================================
# HELPERS — PLOTS
# ============================================================

def plot_cm(y_true, y_pred, title, fname):
    cm     = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    annot  = np.array([[f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)"
                        for j in range(2)] for i in range(2)])
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues',
                linewidths=0.5, linecolor='white', ax=ax,
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'],
                cbar_kws={'label': 'Count'})
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    tn, fp, fn, tp = cm.ravel()
    acc  = (tp + tn) / (tp + tn + fp + fn)
    sens = tp / (tp + fn) if (tp + fn) else 0
    spec = tn / (tn + fp) if (tn + fp) else 0
    ax.text(0.5, -0.14,
            f"Acc={acc:.3f}   Sensitivity={sens:.3f}   Specificity={spec:.3f}",
            transform=ax.transAxes, ha='center', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='gray'))
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✓ {fname}")


def plot_roc(y_true, y_prob, title, fname):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    auc     = roc_auc_score(y_true, y_prob)
    best_ix = np.argmin(np.sqrt(fpr**2 + (1 - tpr)**2))
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.fill_between(fpr, tpr, alpha=0.15, color='darkorange')
    ax.plot(fpr, tpr, color='darkorange', lw=2,
            label=f'SVM RBF  AUC={auc:.4f}')
    ax.plot([0, 1], [0, 1], 'navy', lw=1.5, linestyle='--', label='Random')
    ax.scatter(fpr[best_ix], tpr[best_ix], marker='*', s=200,
               color='red', zorder=5,
               label=f'Best thr≈{thr[best_ix]:.3f}')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel('False Positive Rate (1−Specificity)', fontsize=11)
    ax.set_ylabel('True Positive Rate (Sensitivity)',    fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✓ {fname}")


def plot_learning_curve(loop_log):
    """
    Plot train F1 vs val F1 across training rounds.
    Highlights the best round and any overfitted rounds.
    """
    rounds      = [r['loop']      for r in loop_log]
    train_f1s   = [r['train_f1']  for r in loop_log]
    val_f1s     = [r['val_f1']    for r in loop_log]
    gaps        = [r['gap']       for r in loop_log]
    best_round  = max(loop_log, key=lambda r: r['val_f1'])['loop']

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

    # ── Top: Train vs Val F1 ──────────────────────────────
    ax1.plot(rounds, train_f1s, 'o-', color='steelblue',
             lw=2, label='Train F1')
    ax1.plot(rounds, val_f1s,   's-', color='darkorange',
             lw=2, label='Validation F1')
    ax1.axvline(best_round, color='green', linestyle='--', lw=1.5,
                label=f'Best round={best_round}')

    # shade overfit rounds
    for r in loop_log:
        if r['overfit_flag']:
            ax1.axvspan(r['loop'] - 0.4, r['loop'] + 0.4,
                        alpha=0.15, color='red')

    ax1.set_ylabel('F1 Score', fontsize=11)
    ax1.set_ylim([0, 1.1])
    ax1.set_title('Training Loop — F1 Score per Round\n'
                  '(red shading = overfit flag: train-val gap > '
                  f'{OVERFIT_THRESHOLD})',
                  fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)

    overfit_patch = mpatches.Patch(color='red', alpha=0.3,
                                   label='Overfit flag')
    ax1.legend(fontsize=10, handles=ax1.get_legend_handles_labels()[0]
               + [overfit_patch])

    # ── Bottom: Train - Val gap ───────────────────────────
    bar_colors = ['#e74c3c' if g > OVERFIT_THRESHOLD
                  else '#2ecc71' for g in gaps]
    ax2.bar(rounds, gaps, color=bar_colors, edgecolor='black',
            alpha=0.8, width=0.6)
    ax2.axhline(OVERFIT_THRESHOLD, color='red', linestyle='--',
                lw=1.5, label=f'Overfit threshold ({OVERFIT_THRESHOLD})')
    for i, (r, g) in enumerate(zip(rounds, gaps)):
        ax2.text(r, g + 0.002, f'{g:.3f}', ha='center',
                 fontsize=8, fontweight='bold')
    ax2.set_xlabel('Round', fontsize=11)
    ax2.set_ylabel('Train − Val F1 Gap', fontsize=11)
    ax2.set_title('Overfitting Monitor — Train/Val F1 Gap per Round',
                  fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.set_xticks(rounds)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'svm_learning_curve.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("  ✓ svm_learning_curve.png")


def plot_cv_bars(cv_res):
    metrics_list = ['accuracy', 'precision', 'recall', 'f1']
    labels       = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors       = ['steelblue', 'darkorange', 'seagreen', 'crimson']
    folds        = [f'Fold {i+1}' for i in range(N_FOLDS)]
    fig, axes    = plt.subplots(2, 2, figsize=(14, 9))
    for ax, met, lab, col in zip(axes.ravel(), metrics_list, labels, colors):
        sc = cv_res[f'test_{met}']
        ax.bar(folds, sc, color=col, alpha=0.75, edgecolor='black')
        ax.axhline(sc.mean(), color='black', linestyle='--', lw=1.5,
                   label=f'Mean={sc.mean():.4f} ± {sc.std():.4f}')
        for i, v in enumerate(sc):
            ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
        ax.set_title(f'5-Fold CV — {lab}', fontsize=12, fontweight='bold')
        ax.set_ylim([0, 1.15])
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.3)
    plt.suptitle('5-Fold Cross Validation — SVM RBF (Best Round)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'svm_cv_results.png'), dpi=150)
    plt.show()
    print("  ✓ svm_cv_results.png")


def plot_metrics_summary(val_m, test_m):
    names = ['Accuracy', 'Precision', 'Recall\n(Sensitivity)',
             'Specificity', 'F1-Score', 'ROC-AUC']
    keys  = ['accuracy', 'precision', 'recall_sensitivity',
             'specificity', 'f1_score', 'roc_auc']
    vv = [val_m[k]  for k in keys]
    tv = [test_m[k] for k in keys]
    x  = np.arange(len(names)); w = 0.35
    fig, ax = plt.subplots(figsize=(13, 6))
    b1 = ax.bar(x - w/2, vv, w, label='Validation (best round)',
                color='steelblue',  alpha=0.8, edgecolor='black')
    b2 = ax.bar(x + w/2, tv, w, label='Test Set (final)',
                color='darkorange', alpha=0.8, edgecolor='black')
    for b in list(b1) + list(b2):
        h = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 1.15])
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'All Evaluation Metrics — SVM RBF '
                 f'(C={SVM_C}, gamma={SVM_GAMMA})',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'svm_metrics_summary.png'), dpi=150)
    plt.show()
    print("  ✓ svm_metrics_summary.png")


def plot_overfit_summary(loop_log):
    """
    Scatter: each round plotted as train_f1 vs val_f1.
    Points above the diagonal are "good" (no overfit).
    """
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5,
            label='Perfect generalisation (train=val)')
    ax.fill_between([0, 1], [0, 1], [0+OVERFIT_THRESHOLD, 1+OVERFIT_THRESHOLD],
                    alpha=0.07, color='red',
                    label=f'Overfit zone (gap>{OVERFIT_THRESHOLD})')

    for r in loop_log:
        color  = '#e74c3c' if r['overfit_flag'] else '#2ecc71'
        marker = 'X' if r['overfit_flag'] else 'o'
        ax.scatter(r['val_f1'], r['train_f1'], c=color,
                   marker=marker, s=120, zorder=5,
                   edgecolors='black', linewidths=0.6)
        ax.annotate(f"R{r['loop']}", (r['val_f1'], r['train_f1']),
                    textcoords='offset points', xytext=(5, 4), fontsize=8)

    good_patch  = mpatches.Patch(color='#2ecc71', label='Healthy round')
    overfit_pat = mpatches.Patch(color='#e74c3c', label='Overfit round')
    ax.legend(handles=[good_patch, overfit_pat] +
              ax.get_legend_handles_labels()[0][1:],
              fontsize=9, loc='upper left')
    ax.set_xlim([0, 1.05]); ax.set_ylim([0, 1.05])
    ax.set_xlabel('Validation F1', fontsize=11)
    ax.set_ylabel('Train F1', fontsize=11)
    ax.set_title('Overfit Scatter — Train F1 vs Val F1 per Round',
                 fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'svm_overfit_scatter.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("  ✓ svm_overfit_scatter.png")


# ============================================================
# STEP 1 — LOAD IMAGES
# ============================================================
print("\n[1] LOADING IMAGES")
print("-"*65)
all_images, all_labels = [], []

print("\n  ── Pneumonia (label=1) ──")
imgs, lbls = load_images(PNEUMONIA_PATH, label=1)
all_images.extend(imgs); all_labels.extend(lbls)

print("\n  ── Normal (label=0) ──")
imgs, lbls = load_images(NORMAL_PATH, label=0)
all_images.extend(imgs); all_labels.extend(lbls)

print(f"\n  Total     : {len(all_images)}")
print(f"  Normal    : {all_labels.count(0)}")
print(f"  Pneumonia : {all_labels.count(1)}")

if len(set(all_labels)) < 2:
    raise RuntimeError("Only one class loaded — check paths.")

# ============================================================
# STEP 2 — FEATURE EXTRACTION
# ============================================================
print("\n[2] EXTRACTING FEATURES")
print("-"*65)
X_raw = extract_features(all_images)
y     = np.array(all_labels)
print(f"  Raw feature shape: {X_raw.shape}")

# ============================================================
# STEP 3 — FIXED TEST SPLIT  (locked for the whole pipeline)
# ============================================================
print("\n[3] CREATING FIXED TEST SPLIT  (seed=42, LOCKED)")
print("-"*65)
# Pull off test set once — it will NEVER be touched until Step 11
X_pool, X_test_raw, y_pool, y_test = train_test_split(
    X_raw, y, test_size=TEST_RATIO,
    stratify=y, random_state=RANDOM_STATE)
print(f"  Pool (train+val) : {len(X_pool)}")
print(f"  Test (LOCKED)    : {len(X_test_raw)}  "
      f"(Normal={sum(y_test==0)}, Pneumonia={sum(y_test==1)})")
print("  ✓ Test set locked. Will NOT be seen until Step 11.")

# ============================================================
# STEP 4 — STANDARDSCALER + PCA  (fit on first round,
#           then reused — prevents leakage)
# ============================================================
# We fit scaler & PCA once on a representative 70% sample
# so that they are consistent across all 20 loops.
print("\n[4] FIT SCALER & PCA ON REPRESENTATIVE TRAINING SAMPLE")
print("-"*65)
X_repr, _, y_repr, _ = train_test_split(
    X_pool, y_pool, test_size=0.30,
    stratify=y_pool, random_state=RANDOM_STATE)

scaler_global = StandardScaler()
X_repr_sc     = scaler_global.fit_transform(X_repr)

pca_global    = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
pca_global.fit(X_repr_sc)

var_explained = sum(pca_global.explained_variance_ratio_) * 100
print(f"  Scaler fit on {len(X_repr)} samples")
print(f"  PCA  {N_COMPONENTS} components  →  {var_explained:.2f}% variance")

# Transform test set (locked — done once here, never again)
X_test_sc  = scaler_global.transform(X_test_raw)
X_test_pca = pca_global.transform(X_test_sc)

# Transform full pool
X_pool_sc  = scaler_global.transform(X_pool)
X_pool_pca = pca_global.transform(X_pool_sc)

# ============================================================
# STEP 5 — 20-ROUND TRAINING LOOP WITH EARLY STOPPING
# ============================================================
print("\n[5] 20-ROUND TRAINING LOOP")
print(f"    Max rounds : {MAX_LOOPS}")
print(f"    Patience   : {PATIENCE}  (stop if no val-F1 improvement)")
print(f"    Overfit guard : flag round if train-val gap > {OVERFIT_THRESHOLD}")
print("-"*65)

best_val_f1    = -np.inf
no_improve_cnt = 0
best_model     = None
best_round     = 0
best_val_metrics = None
loop_log       = []

print(f"\n  {'Round':>6} {'Train-F1':>10} {'Val-F1':>10} "
      f"{'Gap':>8} {'Overfit':>8} {'Status':>14}")
print("  " + "─"*62)

for loop in range(1, MAX_LOOPS + 1):
    # Each round uses a different random_state for the train/val
    # split so the model sees different orderings — simulates
    # repeated cross-validation bootstrapping.
    loop_seed = RANDOM_STATE + loop * 7   # deterministic but varied

    X_tr, X_v, y_tr, y_v = train_test_split(
        X_pool_pca, y_pool,
        test_size=VAL_RATIO / (TRAIN_RATIO + VAL_RATIO),
        stratify=y_pool, random_state=loop_seed)

    # Train SVM
    svm = SVC(kernel=SVM_KERNEL, C=SVM_C, gamma=SVM_GAMMA,
              probability=True, random_state=RANDOM_STATE)
    svm.fit(X_tr, y_tr)

    # Train F1 (overfit check)
    y_tr_pred  = svm.predict(X_tr)
    train_f1   = float(f1_score(y_tr, y_tr_pred, zero_division=0))

    # Val F1
    y_v_pred   = svm.predict(X_v)
    y_v_prob   = svm.predict_proba(X_v)[:, 1]
    val_f1     = float(f1_score(y_v, y_v_pred, zero_division=0))

    gap          = train_f1 - val_f1
    overfit_flag = gap > OVERFIT_THRESHOLD

    loop_log.append(dict(loop=loop, train_f1=train_f1,
                         val_f1=val_f1, gap=gap,
                         overfit_flag=overfit_flag))

    # Improvement check — overfit rounds are NOT accepted as best
    if val_f1 > best_val_f1 and not overfit_flag:
        improvement = val_f1 - best_val_f1
        best_val_f1 = val_f1
        no_improve_cnt = 0
        best_model  = deepcopy(svm)
        best_round  = loop
        best_X_v, best_y_v, best_y_v_prob = X_v, y_v, y_v_prob
        status = "★ NEW BEST"
    else:
        no_improve_cnt += 1
        status = f"no-impr ({no_improve_cnt}/{PATIENCE})"
        if overfit_flag:
            status = f"OVERFIT  no-impr ({no_improve_cnt}/{PATIENCE})"

    flag_str = "YES ⚠" if overfit_flag else "no"
    print(f"  {loop:>6}  {train_f1:>10.4f}  {val_f1:>10.4f}  "
          f"{gap:>8.4f}  {flag_str:>8}  {status:>14}")

    # Early stopping
    if no_improve_cnt >= PATIENCE:
        print(f"\n  ⏹  Early stop at round {loop} "
              f"(no improvement for {PATIENCE} consecutive rounds)")
        break

print(f"\n  ✓ Training complete.  Best round = {best_round}  "
      f"(Val-F1 = {best_val_f1:.4f})")

if best_model is None:
    raise RuntimeError(
        "All rounds were flagged as overfit — try reducing C or "
        "increasing OVERFIT_THRESHOLD.")

# ============================================================
# STEP 6 — VALIDATE BEST MODEL (val split from best round)
# ============================================================
print("\n[6] BEST-ROUND VALIDATION METRICS")
print("-"*65)
best_val_m = compute_metrics(best_y_v, best_model.predict(best_X_v),
                              best_y_v_prob)
print_metrics_table(best_val_m,
                    f"Validation — Round {best_round} (best)")

# ============================================================
# STEP 7 — 5-FOLD CROSS VALIDATION on pool
# ============================================================
print(f"\n[7] 5-FOLD CROSS VALIDATION  (Pool set, best model config)")
print("-"*65)
skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                          random_state=RANDOM_STATE)
svm_cv = SVC(kernel=SVM_KERNEL, C=SVM_C, gamma=SVM_GAMMA,
             probability=True, random_state=RANDOM_STATE)
cv_res = cross_validate(
    svm_cv, X_pool_pca, y_pool, cv=skf,
    scoring={'accuracy': 'accuracy', 'precision': 'precision',
             'recall': 'recall', 'f1': 'f1'},
    return_train_score=True, n_jobs=-1)

print(f"\n  {'Fold':<6} {'Train-F1':>10} {'Val-F1':>10} "
      f"{'Gap':>8} {'Overfit?':>10}")
print("  " + "─"*50)
for i in range(N_FOLDS):
    tr_f1 = cv_res['train_f1'][i]
    vl_f1 = cv_res['test_f1'][i]
    gap   = tr_f1 - vl_f1
    flag  = "YES ⚠" if gap > OVERFIT_THRESHOLD else "no"
    print(f"  Fold {i+1}  {tr_f1:>10.4f}  {vl_f1:>10.4f}  "
          f"{gap:>8.4f}  {flag:>10}")

print("  " + "─"*50)
cv_tr_mean = cv_res['train_f1'].mean()
cv_vl_mean = cv_res['test_f1'].mean()
print(f"  Mean    {cv_tr_mean:>10.4f}  {cv_vl_mean:>10.4f}  "
      f"{cv_tr_mean - cv_vl_mean:>8.4f}")
print(f"  Std     {cv_res['train_f1'].std():>10.4f}  "
      f"{cv_res['test_f1'].std():>10.4f}")

overall_gap = cv_tr_mean - cv_vl_mean
if overall_gap > OVERFIT_THRESHOLD:
    print(f"\n  ⚠  WARNING: CV mean train-val gap ({overall_gap:.4f}) "
          f"exceeds overfit threshold ({OVERFIT_THRESHOLD}).")
    print("     Consider reducing C or increasing PCA components.")
else:
    print(f"\n  ✓ No systematic overfitting detected in CV "
          f"(mean gap = {overall_gap:.4f}).")

# ============================================================
# STEP 8 — LEARNING CURVE & OVERFIT GRAPHS
# ============================================================
print("\n[8] GENERATING DIAGNOSTIC GRAPHS")
print("-"*65)
plot_learning_curve(loop_log)
plot_overfit_summary(loop_log)
plot_cv_bars(cv_res)

# ============================================================
# STEP 9 — LOOP SUMMARY TABLE
# ============================================================
print("\n[9] TRAINING LOOP SUMMARY")
print("-"*65)
overfit_rounds = [r['loop'] for r in loop_log if r['overfit_flag']]
healthy_rounds = [r['loop'] for r in loop_log if not r['overfit_flag']]
total_rounds   = len(loop_log)
print(f"  Total rounds run   : {total_rounds}")
print(f"  Healthy rounds     : {len(healthy_rounds)}  "
      f"→ {healthy_rounds}")
print(f"  Overfit rounds     : {len(overfit_rounds)}  "
      f"→ {overfit_rounds if overfit_rounds else 'none'}")
print(f"  Best round         : {best_round}  "
      f"(Val-F1 = {best_val_f1:.4f})")
print(f"  Early stop         : "
      f"{'YES' if total_rounds < MAX_LOOPS else 'NO (reached MAX_LOOPS)'}")

# ============================================================
# STEP 10 — VALIDATION GRAPHS (best model, val split)
# ============================================================
print("\n[10] VALIDATION GRAPHS  (Best round)")
print("-"*65)
plot_cm(best_y_v, best_model.predict(best_X_v),
        f'Confusion Matrix — SVM RBF  Validation (Round {best_round})',
        'svm_cm_validation.png')
plot_roc(best_y_v, best_y_v_prob,
         f'ROC Curve — SVM RBF  Validation (Round {best_round})',
         'svm_roc_validation.png')

# ============================================================
# STEP 11 — FINAL TEST SET EVALUATION  (test set unlocked)
# ============================================================
print("\n[11] FINAL TEST SET EVALUATION  ← TEST SET UNLOCKED")
print("-"*65)
y_test_pred = best_model.predict(X_test_pca)
y_test_prob = best_model.predict_proba(X_test_pca)[:, 1]
test_m      = compute_metrics(y_test, y_test_pred, y_test_prob)
print_metrics_table(test_m, "Test Set — Final Evaluation")

# Overfit check: compare best val F1 with test F1
final_gap = best_val_f1 - test_m['f1_score']
print(f"\n  Val F1  = {best_val_f1:.4f}")
print(f"  Test F1 = {test_m['f1_score']:.4f}")
print(f"  Gap     = {final_gap:.4f}  "
      f"({'⚠ potential overfit' if final_gap > OVERFIT_THRESHOLD else '✓ within threshold'})")

print(f"\n  Classification Report (Test Set):")
print(classification_report(y_test, y_test_pred,
                             target_names=['Normal', 'Pneumonia']))

print("\n  Test Set Graphs:")
plot_cm(y_test, y_test_pred,
        f'Confusion Matrix Heatmap — SVM RBF  Test Set',
        'svm_cm_test.png')
plot_roc(y_test, y_test_prob,
         f'ROC Curve — SVM RBF  Test Set',
         'svm_roc_test.png')
plot_metrics_summary(best_val_m, test_m)

# ============================================================
# STEP 12 — SAVE
# ============================================================
print("\n[12] SAVING")
print("-"*65)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

model_pkg = dict(
    svm_model     = best_model,
    scaler        = scaler_global,
    pca           = pca_global,
    img_size      = IMG_SIZE,
    n_components  = N_COMPONENTS,
    kernel        = SVM_KERNEL,
    C             = SVM_C,
    gamma         = SVM_GAMMA,
    best_round    = best_round,
    best_val_f1   = best_val_f1,
    training_date = ts,
)
for p in [f'{OUT}/svm_rbf_{ts}.pkl', f'{OUT}/svm_rbf_latest.pkl']:
    with open(p, 'wb') as f:
        pickle.dump(model_pkg, f)
print("  ✓ Model pkl saved")

results_json = {
    'timestamp'    : ts,
    'model'        : f'SVM | kernel={SVM_KERNEL} | C={SVM_C} | gamma={SVM_GAMMA}',
    'split'        : {'train': TRAIN_RATIO, 'val': VAL_RATIO, 'test': TEST_RATIO},
    'pca_variance' : float(var_explained / 100),
    'training_loop': {
        'max_loops'      : MAX_LOOPS,
        'patience'       : PATIENCE,
        'total_rounds'   : total_rounds,
        'best_round'     : best_round,
        'best_val_f1'    : best_val_f1,
        'overfit_rounds' : overfit_rounds,
        'per_round'      : loop_log,
    },
    'cv_5fold' : {
        'mean_train_f1' : float(cv_res['train_f1'].mean()),
        'mean_val_f1'   : float(cv_res['test_f1'].mean()),
        'mean_gap'      : float(cv_tr_mean - cv_vl_mean),
        'mean_accuracy' : float(cv_res['test_accuracy'].mean()),
        'std_accuracy'  : float(cv_res['test_accuracy'].std()),
        'mean_precision': float(cv_res['test_precision'].mean()),
        'mean_recall'   : float(cv_res['test_recall'].mean()),
        'mean_f1'       : float(cv_res['test_f1'].mean()),
        'std_f1'        : float(cv_res['test_f1'].std()),
        'per_fold'      : [{
            'fold'      : i + 1,
            'train_f1'  : float(cv_res['train_f1'][i]),
            'val_f1'    : float(cv_res['test_f1'][i]),
            'gap'       : float(cv_res['train_f1'][i] - cv_res['test_f1'][i]),
            'accuracy'  : float(cv_res['test_accuracy'][i]),
            'precision' : float(cv_res['test_precision'][i]),
            'recall'    : float(cv_res['test_recall'][i]),
        } for i in range(N_FOLDS)],
    },
    'validation_metrics' : best_val_m,
    'test_metrics'       : test_m,
    'overfit_check'      : {
        'val_f1'   : best_val_f1,
        'test_f1'  : test_m['f1_score'],
        'gap'      : final_gap,
        'flagged'  : final_gap > OVERFIT_THRESHOLD,
    },
}
with open(f'{OUT}/svm_rbf_results_{ts}.json', 'w') as f:
    json.dump(results_json, f, indent=4)
print("  ✓ JSON results saved")

with open(f'{OUT}/svm_rbf_report_{ts}.txt', 'w') as f:
    f.write("="*65 + "\n  PNEUMONIA DETECTION — SVM RBF REPORT\n" + "="*65 + "\n\n")
    f.write(f"Date          : {ts}\n")
    f.write(f"Model         : SVM | kernel={SVM_KERNEL} | C={SVM_C} | gamma={SVM_GAMMA}\n")
    f.write(f"Split         : Train 70% | Val 15% | Test 15%\n")
    f.write(f"PCA variance  : {var_explained:.2f}%\n\n")
    f.write("TRAINING LOOP\n" + "─"*40 + "\n")
    f.write(f"  Max rounds   : {MAX_LOOPS}\n")
    f.write(f"  Patience     : {PATIENCE}\n")
    f.write(f"  Total run    : {total_rounds}\n")
    f.write(f"  Best round   : {best_round}  Val-F1={best_val_f1:.4f}\n")
    f.write(f"  Overfit rnds : {overfit_rounds}\n\n")
    f.write("  Round  TrainF1   ValF1     Gap    Overfit\n")
    for r in loop_log:
        f.write(f"  {r['loop']:>5}  {r['train_f1']:.4f}    "
                f"{r['val_f1']:.4f}    {r['gap']:.4f}   "
                f"{'YES' if r['overfit_flag'] else 'no'}\n")
    f.write("\n5-FOLD CV\n" + "─"*40 + "\n")
    for i in range(N_FOLDS):
        f.write(f"  Fold {i+1}  TrainF1={cv_res['train_f1'][i]:.4f}  "
                f"ValF1={cv_res['test_f1'][i]:.4f}\n")
    f.write(f"  Mean gap : {cv_tr_mean - cv_vl_mean:.4f}\n\n")
    for section, m in [("VALIDATION (best round)", best_val_m),
                        ("TEST SET (final)", test_m)]:
        f.write(f"{section}\n" + "─"*40 + "\n")
        for key, val in m.items():
            f.write(f"  {key:<22}: {val}\n")
        f.write("\n")
    f.write("OVERFIT CHECK\n" + "─"*40 + "\n")
    f.write(f"  Val F1  : {best_val_f1:.4f}\n")
    f.write(f"  Test F1 : {test_m['f1_score']:.4f}\n")
    f.write(f"  Gap     : {final_gap:.4f}  "
            f"({'FLAGGED' if final_gap > OVERFIT_THRESHOLD else 'OK'})\n")
print("  ✓ Text report saved")

print("\n  Output files in /kaggle/working:")
for fname in sorted(os.listdir(OUT)):
    fp = os.path.join(OUT, fname)
    if os.path.isfile(fp):
        print(f"    {fname}  ({os.path.getsize(fp)/1024:.1f} KB)")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*65)
print("  FINAL SUMMARY — SVM RBF")
print("="*65)
print(f"  Hyperparameters : kernel={SVM_KERNEL}  C={SVM_C}  gamma={SVM_GAMMA}")
print(f"  Scaler          : StandardScaler (mandatory)")
print(f"  PCA             : {N_COMPONENTS} components  ({var_explained:.2f}% variance)\n")

print("  ┌─ Training Loop ─────────────────────────────────────┐")
print(f"  │  Rounds run      : {total_rounds}/{MAX_LOOPS}")
print(f"  │  Healthy rounds  : {len(healthy_rounds)}")
print(f"  │  Overfit rounds  : {len(overfit_rounds)}  "
      f"{overfit_rounds if overfit_rounds else '(none)'}")
print(f"  │  Best round      : {best_round}  (Val-F1={best_val_f1:.4f})")
print("  └─────────────────────────────────────────────────────┘\n")

print("  ┌─ 5-Fold CV (Pool) ──────────────────────────────────┐")
for met in ['accuracy', 'precision', 'recall', 'f1']:
    v = cv_res[f'test_{met}']
    print(f"  │  {met.capitalize():<12}: {v.mean():.4f} ± {v.std():.4f}")
print(f"  │  Train-Val gap   : {cv_tr_mean - cv_vl_mean:.4f}  "
      f"({'⚠ overfit' if cv_tr_mean - cv_vl_mean > OVERFIT_THRESHOLD else '✓ OK'})")
print("  └─────────────────────────────────────────────────────┘\n")

print("  ┌─ Test Set Evaluation ───────────────────────────────┐")
for name, key in [('Accuracy',             'accuracy'),
                   ('Precision',            'precision'),
                   ('Recall (Sensitivity)', 'recall_sensitivity'),
                   ('Specificity',          'specificity'),
                   ('F1-Score',             'f1_score'),
                   ('ROC-AUC',              'roc_auc')]:
    v = test_m[key]
    print(f"  │  {name:<25} {v:.4f}  ({v*100:.2f}%)")
print("  └─────────────────────────────────────────────────────┘\n")

print("  ┌─ Overfit Report ────────────────────────────────────┐")
print(f"  │  Best Val F1     : {best_val_f1:.4f}")
print(f"  │  Test F1         : {test_m['f1_score']:.4f}")
print(f"  │  Gap             : {final_gap:.4f}  "
      f"({'⚠ FLAGGED' if final_gap > OVERFIT_THRESHOLD else '✓ within threshold'})")
print("  └─────────────────────────────────────────────────────┘\n")

print("  Saved graphs:")
for g in ['svm_learning_curve.png', 'svm_overfit_scatter.png',
          'svm_cv_results.png', 'svm_cm_validation.png',
          'svm_roc_validation.png', 'svm_cm_test.png',
          'svm_roc_test.png', 'svm_metrics_summary.png']:
    print(f"    ✓ {g}")
print("\n  ✓ PIPELINE COMPLETE")
print("="*65)